In [1]:
import pandas as pd
import numpy as np


# load dataset

In [2]:
df = pd.read_csv("helpdesk_tickets.csv")
df.head()


,Ticket_ID,Date_Created,Date_Resolved,Category,Subcategory,Priority,Status,Assigned_Team,Resolution_Time_Hrs,Description,Escalated
0,1,2023-11-16,2023-11-23,Access,Access denied by auditor policy,Low,Open,Desktop Support,3.56,User reported Access denied by auditor policy....,True
1,2,2023-11-29,2023-12-02,Security,Suspicious DLL injection,Critical,Open,Security Team,4.14,User reported Suspicious DLL injection. Furthe...,False
2,3,2023-11-29,2023-12-06,Security,Suspicious SOC alert,Low,Pending,Security Team,2.98,User reported Suspicious SOC alert. Further in...,True
3,4,2023-11-04,2023-11-09,Software,App freezing,Low,On Hold,Application Support,4.45,User reported App freezing. Further investigat...,False
4,5,2023-11-03,2023-11-10,Access,Access logging failure,Critical,In Progress,Desktop Support,3.49,User reported Access logging failure. Further ...,True


# size of dataset

In [3]:
df.shape 

(1000000, 11)

# dataset inspection

In [4]:
df.info


<bound method DataFrame.info of         Ticket_ID Date_Created Date_Resolved  Category  \
0               1   2023-11-16    2023-11-23    Access   
1               2   2023-11-29    2023-12-02  Security   
2               3   2023-11-29    2023-12-06  Security   
3               4   2023-11-04    2023-11-09  Software   
4               5   2023-11-03    2023-11-10    Access   
...           ...          ...           ...       ...   
999995     999996   2023-11-25    2023-11-29  Security   
999996     999997   2023-11-12    2023-11-14  Hardware   
999997     999998   2023-11-05    2023-11-12  Hardware   
999998     999999   2023-11-18           NaN  Software   
999999    1000000   2023-11-19    2023-11-23  Software   

                            Subcategory  Priority       Status  \
0       Access denied by auditor policy       Low         Open   
1              Suspicious DLL injection  Critical         Open   
2                  Suspicious SOC alert       Low      Pending   
3      

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   Ticket_ID            1000000 non-null  int64  
 1   Date_Created         1000000 non-null  object 
 2   Date_Resolved        700057 non-null   object 
 3   Category             1000000 non-null  object 
 4   Subcategory          1000000 non-null  object 
 5   Priority             1000000 non-null  object 
 6   Status               1000000 non-null  object 
 7   Assigned_Team        1000000 non-null  object 
 8   Resolution_Time_Hrs  1000000 non-null  float64
 9   Description          1000000 non-null  object 
 10  Escalated            1000000 non-null  bool   
dtypes: bool(1), float64(1), int64(1), object(8)
memory usage: 77.2+ MB


# data cleaning

In [6]:
df["date_created"] = pd.to_datetime(df["Date_Created"])
df["date_resolved"] = pd.to_datetime(df["Date_Resolved"])

In [7]:
df[["date_created" , "date_resolved"]].head()

,date_created,date_resolved
0,2023-11-16,2023-11-23
1,2023-11-29,2023-12-02
2,2023-11-29,2023-12-06
3,2023-11-04,2023-11-09
4,2023-11-03,2023-11-10


In [8]:
df = df.dropna(subset=["date_resolved"])

In [9]:
df.shape

(700057, 13)

In [10]:
df["calculated_time"] = (
    df["date_resolved"] - df["date_created"]
).dt.total_seconds() / 3600

C:\Users\getru\AppData\Local\Temp\ipykernel_21524\3276081712.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["calculated_time"] = (


In [11]:
df = df.dropna(subset = ["date_resolved"]).copy()

In [12]:
df["calculated_time"] = (
    df["date_resolved"] - df["date_created"]
).dt.total_seconds() / 3600

In [13]:
df[["resolution_time_hrs", "calculated_time"]].head()

KeyError: "['resolution_time_hrs'] not in index"

In [ ]:
df.columns = df.columns.str.lower().str.strip()

In [ ]:
df[["resolution_time_hrs", "calculated_time"]].head()

,resolution_time_hrs,calculated_time
0,3.56,168.0
1,4.14,72.0
2,2.98,168.0
3,4.45,120.0
4,3.49,168.0


# remove helper column

In [ ]:
df = df.drop(columns = ["calculated_time"])

In [ ]:
df.columns

Index(['ticket_id', 'date_created', 'date_resolved', 'category', 'subcategory',
       'priority', 'status', 'assigned_team', 'resolution_time_hrs',
       'description', 'escalated', 'date_created', 'date_resolved'],
      dtype='object')

# remove duplicated columns

In [ ]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
df.columns

Index(['Ticket_ID', 'Date_Created', 'Date_Resolved', 'Category', 'Subcategory',
       'Priority', 'Status', 'Assigned_Team', 'Resolution_Time_Hrs',
       'Description', 'Escalated', 'date_created', 'date_resolved',
       'calculated_time', 'complexity'],
      dtype='object')

# workload metrics

In [ ]:
df.groupby('assigned_team').size()

assigned_team
Application Support    140666
Desktop Support        279475
Network Team           139950
Security Team          139966
dtype: int64

# total workload

In [ ]:
df.groupby('assigned_team')['resolution_time_hrs'].sum()

assigned_team
Application Support    13468560.0
Desktop Support        26859024.0
Network Team           13425600.0
Security Team          13428336.0
Name: resolution_time_hrs, dtype: float64

In [ ]:
# build workload metrics table

In [ ]:
metrics =df.groupby('assigned_team').agg({
    'ticket_id': 'count',
    'resolution_time_hrs' : ['sum','mean'],
    'escalated': 'mean'
}).reset_index()

In [ ]:
metrics

assigned_team ticket_id resolution_time_hrs            escalated
                           count                 sum       mean      mean
0  Application Support    140666          13468560.0  95.748511  0.500199
1      Desktop Support    279475          26859024.0  96.105283  0.498968
2         Network Team    139950          13425600.0  95.931404  0.499971
3        Security Team    139966          13428336.0  95.939985  0.501215

# fixing multi index columns

In [ ]:
metrics = df.groupby('assigned_team').agg(
    ticket_count=('ticket_id', 'count'),
    total_resolution_time=('resolution_time_hrs', 'sum'),
    avg_resolution_time=('resolution_time_hrs', 'mean'),
    escalation_rate=('escalated', 'mean')
).reset_index()

In [ ]:
metrics


,assigned_team,ticket_count,total_resolution_time,avg_resolution_time,escalation_rate
0,Application Support,140666,13468560.0,95.748511,0.500199
1,Desktop Support,279475,26859024.0,96.105283,0.498968
2,Network Team,139950,13425600.0,95.931404,0.499971
3,Security Team,139966,13428336.0,95.939985,0.501215


In [ ]:
df[['resolution_time_hrs', 'escalated']].head()

,resolution_time_hrs,escalated
0,168.0,True
1,72.0,False
2,168.0,True
3,120.0,False
4,168.0,True


In [ ]:
df.groupby('assigned_team')['resolution_time_hrs'].describe()

,count,mean,std,min,25%,50%,75%,max
assigned_team,,,,,,,,
Application Support,140666.0,95.748511,47.958972,24.0,48.0,96.0,144.0,168.0
Desktop Support,279475.0,96.105283,48.028001,24.0,48.0,96.0,144.0,168.0
Network Team,139950.0,95.931404,48.002781,24.0,48.0,96.0,144.0,168.0
Security Team,139966.0,95.939985,47.998076,24.0,48.0,96.0,144.0,168.0


# checking escalation distribution

In [ ]:
df.groupby('assigned_team')['escalated'].value_counts(normalize=True)

assigned_team        escalated
Application Support  True         0.500199
                     False        0.499801
Desktop Support      False        0.501032
                     True         0.498968
Network Team         False        0.500029
                     True         0.499971
Security Team        True         0.501215
                     False        0.498785
Name: proportion, dtype: float64

In [ ]:
metrics = df.groupby('assigned_team').agg(
    ticket_count=('ticket_id', 'count'),
    avg_resolution_time=('resolution_time_hrs', 'mean'),
    median_resolution_time=('resolution_time_hrs', 'median'),
    max_resolution_time=('resolution_time_hrs', 'max'),
    escalation_rate=('escalated', 'mean')
).reset_index()

# Adding a new column for ticket complexity

In [ ]:
import numpy as np

df['complexity'] = np.random.choice(
    ['Low','Medium','High'],
    size=len(df),
    p=[0.5, 0.3, 0.2]
)

In [ ]:
df.head()

,Ticket_ID,Date_Created,Date_Resolved,Category,Subcategory,Priority,Status,Assigned_Team,Resolution_Time_Hrs,Description,Escalated,date_created,date_resolved,calculated_time,complexity
0,1,2023-11-16,2023-11-23,Access,Access denied by auditor policy,Low,Open,Desktop Support,3.56,User reported Access denied by auditor policy....,True,2023-11-16,2023-11-23,168.0,Low
1,2,2023-11-29,2023-12-02,Security,Suspicious DLL injection,Critical,Open,Security Team,4.14,User reported Suspicious DLL injection. Furthe...,False,2023-11-29,2023-12-02,72.0,High
2,3,2023-11-29,2023-12-06,Security,Suspicious SOC alert,Low,Pending,Security Team,2.98,User reported Suspicious SOC alert. Further in...,True,2023-11-29,2023-12-06,168.0,High
3,4,2023-11-04,2023-11-09,Software,App freezing,Low,On Hold,Application Support,4.45,User reported App freezing. Further investigat...,False,2023-11-04,2023-11-09,120.0,Medium
4,5,2023-11-03,2023-11-10,Access,Access logging failure,Critical,In Progress,Desktop Support,3.49,User reported Access logging failure. Further ...,True,2023-11-03,2023-11-10,168.0,High


In [ ]:
df.columns

Index(['ticket_id', 'date_created', 'date_resolved', 'category', 'subcategory',
       'priority', 'status', 'assigned_team', 'resolution_time_hrs',
       'description', 'escalated', 'calculated_time', 'complexity'],
      dtype='object')

In [ ]:
df[['complexity']].head()

,complexity
0,Low
1,High
2,High
3,Medium
4,High


In [ ]:
df['complexity'].value_counts(normalize=True)

complexity
Low       0.500609
Medium    0.299826
High      0.199565
Name: proportion, dtype: float64

In [ ]:
team_factor = {
    'Application Support': 1.0,
    'Desktop Support': 0.8,
    'Network Team': 1.3,
    'Security Team': 1.5
}

df['resolution_time_hrs'] = df['resolution_time_hrs'] * df['assigned_team'].map(team_factor)

In [ ]:
df[['assigned_team', 'resolution_time_hrs']].head()

,assigned_team,resolution_time_hrs
0,Desktop Support,2.2784
1,Security Team,9.3150
2,Security Team,6.7050
3,Application Support,4.4500
4,Desktop Support,2.2336
